In [ ]:
## Problem statement
##Airlines need to predict passenger spending, identify customers who are likely to purchase promotional offers,
##and understand different customer groups.
##Traditional marketing methods may not identify customer preferences accurately,
##leading to missed sales and inefficient advertising expenditure.

In [1]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    accuracy_score,
    classification_report,
    silhouette_score
)

# Create sample airline customer data
np.random.seed(42)
n = 500

df = pd.DataFrame({
    "travel_frequency": np.random.randint(1, 16, n),
    "days_before_trip": np.random.randint(1, 61, n),
    "travel_days": np.random.randint(1, 15, n),
    "offer_discount": np.random.randint(5, 31, n)
})

# Generate spending values for demonstration
df["spending"] = (
    1500
    + df["travel_frequency"] * 500
    + df["travel_days"] * 250
    - df["days_before_trip"] * 10
    + np.random.normal(0, 1000, n)
).clip(lower=500)

# Generate a sample offer-purchase outcome
purchase_score = (
    0.15 * df["travel_frequency"]
    + 0.06 * df["offer_discount"]
    - 0.02 * df["days_before_trip"]
    + np.random.normal(0, 1, n)
)
df["purchased_offer"] = (
    purchase_score > purchase_score.median()
).astype(int)

print("Dataset preview:")
print(df.head())

# 1. REGRESSION: Predict customer spending
X = df[["travel_frequency", "days_before_trip",
        "travel_days"]]
y = df["spending"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

reg = LinearRegression()
reg.fit(X_train, y_train)
reg_pred = reg.predict(X_test)

print("\n--- REGRESSION ---")
print("Actual spending:", y_test.head().round(2).tolist())
print("Predicted spending:", reg_pred[:5].round(2).tolist())
print("MAE:", round(mean_absolute_error(y_test, reg_pred), 2))
print("R2 score:", round(r2_score(y_test, reg_pred), 3))

# 2. CLASSIFICATION: Predict offer purchase
X = df[["travel_frequency", "days_before_trip",
        "offer_discount"]]
y = df["purchased_offer"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
clf_pred = clf.predict(X_test)

print("\n--- CLASSIFICATION ---")
print("Accuracy:", round(accuracy_score(y_test, clf_pred), 3))
print(classification_report(y_test, clf_pred))

# 3. CLUSTERING: Segment airline customers
features = df[["travel_frequency", "spending",
               "travel_days"]]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["customer_cluster"] = kmeans.fit_predict(scaled_features)

print("\n--- CLUSTERING ---")
print(df.groupby("customer_cluster")[
    ["travel_frequency", "spending", "travel_days"]
].mean().round(2))

print("Silhouette score:",
      round(silhouette_score(scaled_features,
                             df["customer_cluster"]), 3))

print("\nCustomer segments:")
print(df[["travel_frequency", "spending",
          "customer_cluster"]].head(10))


Dataset preview:
   travel_frequency  days_before_trip  travel_days  offer_discount  \
0                 7                50            4              15   
1                 4                25           14              10   
2                13                24            1               6   
3                15                13            1              11   
4                11                60            8              22   

       spending  purchased_offer  
0   5545.592164                0  
1   6119.007870                0  
2   8917.494768                1  
3   9155.894550                1  
4  10265.003234                1  

--- REGRESSION ---
Actual spending: [9696.54, 8543.59, 4054.63, 8028.12, 10880.5]
Predicted spending: [10087.99, 10310.36, 5527.68, 7357.04, 11592.28]
MAE: 886.04
R2 score: 0.794

--- CLASSIFICATION ---
Accuracy: 0.71
              precision    recall  f1-score   support

           0       0.68      0.80      0.73        50
           1       0.76 